# 🔬 Декомпозиция эмбеддингов: что от MolFormer, а что новое

**Постановка.** В прошлом анализе мы показали: contrastive encoder в основном дистиллирует MolFormer (linear R²(mf→emb) ≈ 0.89) — но не полностью. Остаётся ~11% дисперсии emb, которая *не* выводится линейно из MolFormer, и именно она может объяснять наблюдаемое преимущество emb в downstream задачах.

**Цель ноутбука.** Разложить emb на два ортогональных подпространства:

$$\text{emb} = \text{emb}_{\text{distilled}} + \text{emb}_{\text{novel}}$$

и показать:

1. Какие *направления* (канонические компоненты) emb коррелируют с MolFormer и насколько (CCA).
2. Какие emb-координаты группируются вместе и с какими таргетами они скоррелированы (co-activation).
3. Per-component SHAP с пометкой "distilled" / "novel".
4. Финальный ablation: предсказание регрессии **только из novel-части** vs **только из distilled-части** vs целого emb.

**Требуется на диске в `BASE_PATH`:** `Test_molformer_original_biocides.xlsx`, `Data_biocides.xlsx`, `QACs_SMILES_MIC_MBC.xlsx`, `Test_molformer_ood.xlsx`, `scaler_classification.pkl`, `encoder_borderline_smote.pth`.


## 1. Импорты и конфиг

In [ ]:
import os, io, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import pearsonr
from catboost import CatBoostRegressor, Pool
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)
OUT_DIR = BASE / 'decomposition'
OUT_DIR.mkdir(exist_ok=True)
print(f'Output: {OUT_DIR}')

RANDOM_STATE = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]
SHORT_NAMES = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
ORIG_COL_MAP = {sn: TARGET_COLUMNS[i] for i, sn in enumerate(SHORT_NAMES)}
OOD_COL_MAP = {
    'SA_MIC': 'MIC Sa', 'SA_MBC': 'MBC Sa',
    'EC_MIC': 'MIC Ec', 'EC_MBC': 'MBC Ec',
}


## 2. Архитектура и загрузка

In [ ]:
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 256],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for hd in hidden_dims:
            layers.extend([nn.Linear(prev, hd), nn.BatchNorm1d(hd),
                           nn.ReLU(), nn.Dropout(dropout)])
            prev = hd
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim), nn.BatchNorm1d(projection_dim),
            nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim), nn.BatchNorm1d(projection_dim),
        )
    def forward(self, x, return_projection=True):
        f = self.encoder(x)
        if return_projection:
            return f, self.projection_head(f)
        return f


def load_excel_safe(path):
    with open(path, 'rb') as f:
        buf = io.BytesIO(f.read())
    return pd.read_excel(buf, engine='openpyxl')


with tqdm(total=6, desc='Loading') as pbar:
    pbar.set_postfix_str('orig molformer')
    data_orig = load_excel_safe(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(['Unnamed: 0'], axis=1)
    pbar.update(1)

    pbar.set_postfix_str('orig targets')
    target_orig = load_excel_safe(f'{BASE_PATH}/Data_biocides.xlsx')
    pbar.update(1)

    pbar.set_postfix_str('ood raw')
    ood_raw = load_excel_safe(f'{BASE_PATH}/QACs_SMILES_MIC_MBC.xlsx')
    pbar.update(1)

    pbar.set_postfix_str('ood molformer')
    ood_molformer = load_excel_safe(f'{BASE_PATH}/Test_molformer_ood.xlsx')
    for c in ['Unnamed: 0']:
        if c in ood_molformer.columns:
            ood_molformer = ood_molformer.drop(columns=[c])
    ood_molformer = ood_molformer.select_dtypes(include=[np.number])
    ood_raw['MBC Ec'] = pd.to_numeric(ood_raw['MBC Ec'], errors='coerce')
    pbar.update(1)

    pbar.set_postfix_str('scaler')
    with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
        scaler_clf = pickle.load(f)
    pbar.update(1)

    pbar.set_postfix_str('encoder')
    encoder_cls = AdvancedContrastiveEncoder(
        input_dim=768, hidden_dims=[512, 384, 256],
        projection_dim=128, dropout=0.3,
    ).to(device)
    encoder_cls.load_state_dict(
        torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
    )
    encoder_cls.eval()
    pbar.update(1)


def extract_emb(enc, X_scaled, batch_size=128):
    enc.eval()
    Xt, out = torch.FloatTensor(X_scaled), []
    with torch.no_grad():
        for i in range(0, len(Xt), batch_size):
            out.append(enc(Xt[i:i+batch_size].to(device),
                           return_projection=False).cpu().numpy())
    return np.concatenate(out, axis=0)


orig_scaled = scaler_clf.transform(data_orig)
ood_scaled  = scaler_clf.transform(ood_molformer)
orig_emb    = extract_emb(encoder_cls, orig_scaled)
ood_emb     = extract_emb(encoder_cls, ood_scaled)
orig_mf     = data_orig.values.astype(float)
ood_mf      = ood_molformer.values.astype(float)

D_MF, D_EMB = orig_mf.shape[1], orig_emb.shape[1]
print(f'orig: mf={orig_mf.shape}, emb={orig_emb.shape}')
print(f'ood:  mf={ood_mf.shape},  emb={ood_emb.shape}')


## 3. Сборка (mol, target) пар для downstream

Для регрессии каждая (молекула, таргет)-пара = одна строка с y = log₂(MIC/MBC) и task_id. Дополнительно вытаскиваем уникальные молекулы для CCA — там нужны фичи без дублирования по таргетам.


In [ ]:
def build_pairs(mf, emb, target_df, col_map):
    rows = []
    for ti, sn in enumerate(SHORT_NAMES):
        col = col_map[sn]
        if col not in target_df.columns: continue
        vals = pd.to_numeric(target_df[col], errors='coerce')
        idx = np.where(vals.notna().values)[0]
        for gi in idx:
            y = float(vals.values[gi])
            if y <= 0: continue
            rows.append({'task': ti, 'sn': sn, 'gi': int(gi),
                         'y_log2': float(np.log2(y)), 'y_raw': y,
                         'mf': mf[gi], 'emb': emb[gi]})
    return rows

def arrs(pairs):
    return {
        'mf':     np.stack([r['mf']  for r in pairs]),
        'emb':    np.stack([r['emb'] for r in pairs]),
        'task':   np.array([r['task'] for r in pairs]),
        'gi':     np.array([r['gi']   for r in pairs]),
        'y_log2': np.array([r['y_log2'] for r in pairs]),
        'sn':     np.array([r['sn']   for r in pairs]),
    }

P_in  = arrs(build_pairs(orig_mf, orig_emb, target_orig, ORIG_COL_MAP))
P_ood = arrs(build_pairs(ood_mf,  ood_emb,  ood_raw,    OOD_COL_MAP))
print(f'in-domain pairs: {len(P_in["task"])}')
print(f'OOD pairs:       {len(P_ood["task"])}')

# для CCA нужны уникальные молекулы (без дублей по таргетам)
_, uniq_in  = np.unique(P_in['gi'],  return_index=True)
_, uniq_ood = np.unique(P_ood['gi'], return_index=True)
MF_in,  EMB_in  = P_in['mf'][uniq_in],   P_in['emb'][uniq_in]
MF_ood, EMB_ood = P_ood['mf'][uniq_ood], P_ood['emb'][uniq_ood]
print(f'unique mols: in={len(MF_in)}, ood={len(MF_ood)}')


## 4. CCA: какие направления emb коррелируют с MolFormer

CCA ищет пары направлений $(u_k, v_k)$ в $\mathbb{R}^{768}$ и $\mathbb{R}^{256}$ соответственно, максимизирующие корреляцию $\rho_k = \text{corr}(\text{MF} \cdot u_k, \text{EMB} \cdot v_k)$. Чем больше $k$, тем меньше $\rho_k$. Направления с $\rho_k \approx 1$ — "дистиллированные"; с маленькой $\rho_k$ — "новые".

**Технически:** CCA через SVD на whitened-данных, чтобы получить все $\min(D_{mf}, D_{emb}) = 256$ канонических корреляций сразу. В лоб через `sklearn.cross_decomposition.CCA` на $D_{mf}=768$ нестабильно при N=108, потому используем ручной устойчивый вариант.


In [ ]:
def whitened_cca(X, Y, eps=1e-6):
    """Канонические корреляции через SVD на whitened данных. Возвращает rho, Wx, Wy."""
    Xc = X - X.mean(0); Yc = Y - Y.mean(0)
    Ux, Sx, Vtx = np.linalg.svd(Xc, full_matrices=False)
    Uy, Sy, Vty = np.linalg.svd(Yc, full_matrices=False)
    rx = int((Sx > eps * Sx.max()).sum())
    ry = int((Sy > eps * Sy.max()).sum())
    Ux, Uy = Ux[:, :rx], Uy[:, :ry]
    Sx, Sy = Sx[:rx], Sy[:ry]
    Vtx, Vty = Vtx[:rx], Vty[:ry]
    M = Ux.T @ Uy
    U, rho, Vt = np.linalg.svd(M, full_matrices=False)
    Wx = Vtx.T @ (U   / Sx[:, None])
    Wy = Vty.T @ (Vt.T / Sy[:, None])
    return rho, Wx, Wy


rho, W_mf, W_emb = whitened_cca(MF_in, EMB_in)
print('Canonical correlations: shape', rho.shape)
print('  rho_1..5  =', rho[:5].round(3))
print('  rho_6..10 =', rho[5:10].round(3))
if len(rho) >= 200:
    print('  rho_50    = {:.3f},  rho_100 = {:.3f},  rho_200 = {:.3f}'
          .format(rho[49], rho[99], rho[199]))


fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(1, len(rho) + 1), rho, lw=1.5, color='#2E75B6')
ax.fill_between(np.arange(1, len(rho) + 1), rho, alpha=0.2, color='#2E75B6')
ax.axhline(0.9, ls='--', c='gray',   alpha=0.6, label='rho = 0.9')
ax.axhline(0.5, ls='--', c='orange', alpha=0.6, label='rho = 0.5')
ax.set_xlabel('Canonical component k', fontweight='bold')
ax.set_ylabel('Canonical correlation rho_k', fontweight='bold')
ax.set_title('Canonical correlation spectrum: MolFormer vs emb (in-domain)', fontweight='bold')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'cca_spectrum.png', dpi=300, bbox_inches='tight')
plt.show()


k_distilled = int((rho > 0.9).sum())
k_partial   = int(((rho > 0.5) & (rho <= 0.9)).sum())
k_novel     = int((rho <= 0.5).sum())
print('\n  high-correlation (rho>0.9):     {} компонент — distilled'.format(k_distilled))
print('  mid-correlation  (0.5<rho<=0.9): {} компонент — shared partial'.format(k_partial))
print('  low-correlation  (rho<=0.5):     {} компонент — novel'.format(k_novel))


## 5. Разложение emb на distilled + novel

`emb_distilled` — линейная проекция emb на подпространство, объяснимое MolFormer (через ridge-регрессию molformer → emb). `emb_novel` — ортогональный остаток.

$$\text{emb}_{\text{distilled}}(x) = \hat{f}_{\text{ridge}}(\text{mf}(x)), \quad \text{emb}_{\text{novel}}(x) = \text{emb}(x) - \text{emb}_{\text{distilled}}(x)$$

Параметры ridge-регрессии фитятся **только на in-domain** молекулах, а применяются и к in-domain, и к OOD. Это даёт честную картину: distilled-часть на OOD — это то, что энкодер "получил бы" из MolFormer-фич, не зная биологии; novel-часть — то, что энкодер добавил сверх.


In [ ]:
scaler_mf = StandardScaler().fit(MF_in)
MF_in_s   = scaler_mf.transform(MF_in)
MF_ood_s  = scaler_mf.transform(MF_ood)

ridge = Ridge(alpha=1.0).fit(MF_in_s, EMB_in)
emb_distilled_in  = ridge.predict(MF_in_s)
emb_distilled_ood = ridge.predict(MF_ood_s)

emb_novel_in  = EMB_in  - emb_distilled_in
emb_novel_ood = EMB_ood - emb_distilled_ood


def explained_var_share(part, whole):
    return (part ** 2).sum() / (whole ** 2).sum()


share_distilled_in  = explained_var_share(emb_distilled_in,  EMB_in)
share_novel_in      = explained_var_share(emb_novel_in,      EMB_in)
share_distilled_ood = explained_var_share(emb_distilled_ood, EMB_ood)
share_novel_ood     = explained_var_share(emb_novel_ood,     EMB_ood)

print('  IN-DOMAIN: distilled={:.3f}, novel={:.3f}, sum={:.3f}'
      .format(share_distilled_in, share_novel_in,
              share_distilled_in + share_novel_in))
print('  OOD:       distilled={:.3f}, novel={:.3f}, sum={:.3f}'
      .format(share_distilled_ood, share_novel_ood,
              share_distilled_ood + share_novel_ood))


# Per-coordinate R²
per_coord_R2 = []
for j in range(D_EMB):
    yhat = emb_distilled_in[:, j]
    y    = EMB_in[:, j]
    ss_res = ((y - yhat) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum() + 1e-12
    per_coord_R2.append(1.0 - ss_res / ss_tot)
per_coord_R2 = np.array(per_coord_R2)

is_distilled = per_coord_R2 >= 0.9
is_partial   = (per_coord_R2 >= 0.5) & (per_coord_R2 < 0.9)
is_novel     = per_coord_R2 < 0.5
print('\nPer-coordinate split:')
print('  distilled (R2 >= 0.9):  {} / {}'.format(int(is_distilled.sum()), D_EMB))
print('  partial   (0.5 <= R2 < 0.9): {} / {}'.format(int(is_partial.sum()), D_EMB))
print('  novel     (R2 < 0.5):  {} / {}'.format(int(is_novel.sum()), D_EMB))


fig, ax = plt.subplots(figsize=(12, 4))
order = np.argsort(per_coord_R2)[::-1]
colors = np.where(per_coord_R2[order] >= 0.9, '#2E75B6',
          np.where(per_coord_R2[order] >= 0.5, '#ED7D31', '#C44E52'))
ax.bar(np.arange(D_EMB), per_coord_R2[order], color=colors, edgecolor='none', width=1.0)
ax.axhline(0.9, ls='--', c='gray', alpha=0.5)
ax.axhline(0.5, ls='--', c='gray', alpha=0.5)
ax.set_xlabel('emb coordinate (sorted by R2)', fontweight='bold')
ax.set_ylabel('R2(MolFormer -> emb_j)', fontweight='bold')
ax.set_title('How much of each emb coordinate is linearly explainable from MolFormer',
             fontweight='bold')
ax.set_xlim(0, D_EMB); ax.set_ylim(0, 1.05)
ax.legend(handles=[
    Patch(color='#2E75B6', label='distilled (R2 >= 0.9): {}'.format(int(is_distilled.sum()))),
    Patch(color='#ED7D31', label='partial (0.5 - 0.9): {}'.format(int(is_partial.sum()))),
    Patch(color='#C44E52', label='novel (R2 < 0.5): {}'.format(int(is_novel.sum()))),
], loc='lower left')
plt.tight_layout()
plt.savefig(OUT_DIR / 'per_coord_R2.png', dpi=300, bbox_inches='tight')
plt.show()


pd.DataFrame({
    'coord': np.arange(D_EMB),
    'R2_from_mf': per_coord_R2,
    'block': np.where(is_distilled, 'distilled',
              np.where(is_partial, 'partial', 'novel'))
}).to_csv(OUT_DIR / 'emb_coord_classification.csv', index=False)


## 6. Co-activation: какие emb-координаты загораются вместе

Для каждой emb-координаты считаем корреляцию её значений (по in-domain молекулам) с каждой другой emb-координатой и с log₂(MIC/MBC) по 4 таргетам. Получаем кластеры co-active координат и их связь с биологической активностью.

- Корреляционная матрица 256×256 (по in-domain)
- Иерархическая кластеризация на $1 - |\rho|$ → выделяем 8 модулей
- Для каждого модуля считаем средний $r$ его активации с 4 таргетами
- И долю distilled / partial / novel координат в каждом модуле — чтобы увидеть, какие модули "про molformer", а какие "про новое знание"


In [ ]:
# средняя активность каждой in-domain молекулы по 4 таргетам (с NaN там где нет данных)
n_in = len(MF_in)
y_mat = np.full((n_in, 4), np.nan)
gi_to_pos = {gi: pos for pos, gi in enumerate(P_in['gi'][uniq_in])}
for r in range(len(P_in['gi'])):
    gi, ti, y = int(P_in['gi'][r]), int(P_in['task'][r]), float(P_in['y_log2'][r])
    pos = gi_to_pos.get(gi)
    if pos is not None:
        y_mat[pos, ti] = y


# Корреляционная матрица emb x emb
emb_corr = np.corrcoef(EMB_in.T)
np.fill_diagonal(emb_corr, 0.0)

dist = 1.0 - np.abs(emb_corr)
np.fill_diagonal(dist, 0.0)
condensed = dist[np.triu_indices(D_EMB, k=1)]
Z = linkage(condensed, method='average')
N_CLUSTERS = 8
labels = fcluster(Z, t=N_CLUSTERS, criterion='maxclust')
print('Кластеров: {}; распределение размеров:'.format(N_CLUSTERS))
print(pd.Series(labels).value_counts().sort_index().to_string())


# Активация модуля = среднее emb-координат, принадлежащих модулю
module_act = np.zeros((n_in, N_CLUSTERS))
for k in range(1, N_CLUSTERS + 1):
    mask = labels == k
    module_act[:, k-1] = EMB_in[:, mask].mean(axis=1)


# Корреляция module x target
mod_target_corr = np.full((N_CLUSTERS, 4), np.nan)
for k in range(N_CLUSTERS):
    for t in range(4):
        valid = ~np.isnan(y_mat[:, t])
        if valid.sum() < 5: continue
        r, _ = pearsonr(module_act[valid, k], y_mat[valid, t])
        mod_target_corr[k, t] = r


# Состав модулей
mod_composition = []
for k in range(1, N_CLUSTERS + 1):
    mask = labels == k
    mod_composition.append({
        'module': k, 'size': int(mask.sum()),
        'distilled': int(is_distilled[mask].sum()),
        'partial':   int(is_partial[mask].sum()),
        'novel':     int(is_novel[mask].sum()),
        'mean_R2_from_mf': float(per_coord_R2[mask].mean()),
    })
mod_comp_df = pd.DataFrame(mod_composition)
print('\nСостав модулей:')
print(mod_comp_df.to_string(index=False))
mod_comp_df.to_csv(OUT_DIR / 'module_composition.csv', index=False)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
im = ax.imshow(mod_target_corr, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')
ax.set_xticks(range(4)); ax.set_xticklabels(SHORT_NAMES, rotation=30, ha='right')
ax.set_yticks(range(N_CLUSTERS)); ax.set_yticklabels(['M{}'.format(k+1) for k in range(N_CLUSTERS)])
ax.set_title('Pearson r(module, log2(activity))', fontweight='bold')
for i in range(N_CLUSTERS):
    for j in range(4):
        v = mod_target_corr[i, j]
        if not np.isnan(v):
            ax.text(j, i, '{:+.2f}'.format(v), ha='center', va='center',
                    color='white' if abs(v) > 0.3 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, label='Pearson r')

ax = axes[1]
bottoms = np.zeros(N_CLUSTERS)
for kind, color in [('distilled', '#2E75B6'), ('partial', '#ED7D31'), ('novel', '#C44E52')]:
    vals = np.array([mod_comp_df.loc[i, kind] for i in range(N_CLUSTERS)])
    ax.bar(np.arange(N_CLUSTERS), vals, bottom=bottoms, color=color, edgecolor='black',
           label=kind, linewidth=0.5)
    bottoms += vals
ax.set_xticks(np.arange(N_CLUSTERS))
ax.set_xticklabels(['M{}'.format(k+1) for k in range(N_CLUSTERS)])
ax.set_ylabel('# emb coordinates', fontweight='bold')
ax.set_title('Module composition (where coordinates come from)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'modules_activity_composition.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Per-component SHAP: какие emb-координаты использует дерево

Обучаем multi-task CatBoost на [emb (256) | task_id] и считаем |SHAP| для каждой emb-координаты на OOD. Группируем SHAP по типу (distilled / partial / novel) и смотрим, что именно дерево считает важным. Дополнительно — scatter «R²(MolFormer → emb_j) vs |SHAP|», чтобы увидеть, есть ли смещение в сторону одной из категорий.


In [ ]:
def make_frame(emb_arr, task):
    df = pd.DataFrame(emb_arr, columns=['emb_{}'.format(i) for i in range(D_EMB)])
    df['task_id'] = task.astype(str)
    return df

df_in  = make_frame(P_in['emb'],  P_in['task'])
df_ood = make_frame(P_ood['emb'], P_ood['task'])

cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                       verbose=False, random_seed=RANDOM_STATE)
cb.fit(Pool(df_in, label=P_in['y_log2'], cat_features=['task_id']))

shap_vals = cb.get_feature_importance(
    Pool(df_ood, cat_features=['task_id']), type='ShapValues'
)[:, :-1]
shap_mean_abs = np.abs(shap_vals).mean(0)
shap_emb  = shap_mean_abs[:D_EMB]
shap_task = shap_mean_abs[D_EMB]


type_arr = np.where(is_distilled, 'distilled',
            np.where(is_partial, 'partial', 'novel'))
shap_by_type = pd.DataFrame({'type': type_arr, 'shap': shap_emb}) \
    .groupby('type').agg(['sum', 'mean', 'count'])
print('SHAP-важность по типам координат (на OOD):')
print(shap_by_type)
shap_by_type.to_csv(OUT_DIR / 'shap_by_type.csv')


top_idx = np.argsort(shap_emb)[::-1][:20]
top_df = pd.DataFrame({
    'coord': top_idx,
    'mean_abs_shap': shap_emb[top_idx],
    'R2_from_mf': per_coord_R2[top_idx],
    'type': type_arr[top_idx],
})
print('\nTop-20 emb-координат по |SHAP| на OOD:')
print(top_df.to_string(index=False))
top_df.to_csv(OUT_DIR / 'top20_emb_coords_shap.csv', index=False)


fig, ax = plt.subplots(figsize=(10, 6))
color_map = {'distilled': '#2E75B6', 'partial': '#ED7D31', 'novel': '#C44E52'}
for t, c in color_map.items():
    m = type_arr == t
    ax.scatter(per_coord_R2[m], shap_emb[m], color=c, label=t,
               edgecolors='black', linewidths=0.4, s=40, alpha=0.75)
ax.axvline(0.9, ls='--', c='gray', alpha=0.5)
ax.axvline(0.5, ls='--', c='gray', alpha=0.5)
ax.set_xlabel('R2(MolFormer -> emb_j)', fontweight='bold')
ax.set_ylabel('Mean |SHAP| on OOD', fontweight='bold')
ax.set_title('Linear explainability from MolFormer vs OOD SHAP importance',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_vs_R2.png', dpi=300, bbox_inches='tight')
plt.show()


## 8. Финальный ablation: распределяем "вину" — distilled или novel?

Обучаем три CatBoost-модели, на одной и той же выборке, с одинаковой архитектурой:

- `emb_full` — целый emb (256d)
- `emb_distilled` — только проекция emb на подпространство MolFormer
- `emb_novel` — только ортогональный остаток

Если **`emb_novel` сам по себе** даёт R² > 0 на OOD — значит "новое знание" в эмбеддингах действительно несёт информацию о биологической активности, не сводимую к MolFormer. Если **`emb_distilled` близок к `emb_full`** — значит добавка от novel-части минимальна.


In [ ]:
distilled_pairs_in  = ridge.predict(scaler_mf.transform(P_in['mf']))
distilled_pairs_ood = ridge.predict(scaler_mf.transform(P_ood['mf']))
novel_pairs_in      = P_in['emb']  - distilled_pairs_in
novel_pairs_ood     = P_ood['emb'] - distilled_pairs_ood


def downstream(emb_in_pairs, emb_ood_pairs, tag):
    df_tr = pd.DataFrame(emb_in_pairs,  columns=['e_{}'.format(i) for i in range(D_EMB)])
    df_tr['task_id'] = P_in['task'].astype(str)
    df_te = pd.DataFrame(emb_ood_pairs, columns=df_tr.columns[:-1])
    df_te['task_id'] = P_ood['task'].astype(str)
    m = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                          verbose=False, random_seed=RANDOM_STATE)
    m.fit(Pool(df_tr, label=P_in['y_log2'], cat_features=['task_id']))
    yp = m.predict(Pool(df_te, cat_features=['task_id']))
    yt = P_ood['y_log2']
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    mae  = float(np.mean(np.abs(yt - yp)))
    r2   = float(r2_score(yt, yp))
    per_target = {}
    for ti, sn in enumerate(SHORT_NAMES):
        mt = P_ood['task'] == ti
        if mt.sum() < 2: continue
        per_target[sn] = {
            'rmse': float(np.sqrt(np.mean((yt[mt] - yp[mt]) ** 2))),
            'mae':  float(np.mean(np.abs(yt[mt] - yp[mt]))),
            'r2':   float(r2_score(yt[mt], yp[mt])) if mt.sum() > 2 else float('nan'),
        }
    return {'tag': tag, 'rmse': rmse, 'mae': mae, 'r2': r2, 'per_target': per_target}


print('Downstream OOD по подпространствам emb')
res_full = downstream(P_in['emb'],         P_ood['emb'],         'emb_full')
res_dist = downstream(distilled_pairs_in,  distilled_pairs_ood,  'emb_distilled')
res_nov  = downstream(novel_pairs_in,      novel_pairs_ood,      'emb_novel')
for r in [res_full, res_dist, res_nov]:
    print('  {:14s}  RMSE={:.3f}  MAE={:.3f}  R2={:.3f}'
          .format(r['tag'], r['rmse'], r['mae'], r['r2']))


rows = []
for r in [res_full, res_dist, res_nov]:
    for sn, m in r['per_target'].items():
        rows.append({'subspace': r['tag'], 'target': sn, **m})
per_t_df = pd.DataFrame(rows)
print('\nPer-target:')
print(per_t_df.round(3).to_string(index=False))
per_t_df.to_csv(OUT_DIR / 'subspace_ablation_per_target.csv', index=False)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
tags = ['emb_full', 'emb_distilled', 'emb_novel']
colors = ['#5DB9E9', '#2E75B6', '#C44E52']
rmse_v = [res_full['rmse'], res_dist['rmse'], res_nov['rmse']]
bars = ax.bar(tags, rmse_v, color=colors, edgecolor='black')
for b, v in zip(bars, rmse_v):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), '{:.3f}'.format(v),
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('OOD RMSE (log2)', fontweight='bold')
ax.set_title('Decomposition ablation: OOD RMSE', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
x = np.arange(4); w = 0.27
for i, (tag, c, r) in enumerate(zip(tags, colors, [res_full, res_dist, res_nov])):
    vals = [r['per_target'].get(sn, {}).get('rmse', float('nan')) for sn in SHORT_NAMES]
    ax.bar(x + (i-1)*w, vals, w, label=tag, color=c, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(SHORT_NAMES)
ax.set_ylabel('OOD RMSE', fontweight='bold')
ax.set_title('Per-target', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'subspace_ablation.png', dpi=300, bbox_inches='tight')
plt.show()


pd.DataFrame([
    {'subspace': r['tag'], 'rmse': r['rmse'], 'mae': r['mae'], 'r2': r['r2']}
    for r in [res_full, res_dist, res_nov]
]).to_csv(OUT_DIR / 'subspace_ablation_overall.csv', index=False)


## 9. Финальный вердикт

In [ ]:
print('=' * 72); print('ВЕРДИКТ: что отвечает за новое знание'); print('=' * 72)

print('\n[A] Линейная объяснимость emb из MolFormer')
print('  Per-coordinate R2:  distilled (>=0.9)={}  partial (0.5-0.9)={}  novel (<0.5)={}'
      .format(int(is_distilled.sum()), int(is_partial.sum()), int(is_novel.sum())))
print('  Доля дисперсии (in): distilled={:.3f}  novel={:.3f}'
      .format(share_distilled_in, share_novel_in))

print('\n[B] SHAP-важность на OOD по типам')
print(shap_by_type)

print('\n[C] Downstream RMSE на OOD')
print('  emb_full       : {:.3f}'.format(res_full['rmse']))
print('  emb_distilled  : {:.3f}'.format(res_dist['rmse']))
print('  emb_novel ONLY : {:.3f}'.format(res_nov ['rmse']))

print('\nИНТЕРПРЕТАЦИЯ:')
print('  novel-часть emb даёт сама по себе RMSE={:.3f}, R2={:.3f}.'
      .format(res_nov['rmse'], res_nov['r2']))
if res_nov['r2'] > 0.05:
    print('  -> novel-часть несёт независимый сигнал о биологической активности.')
else:
    print('  -> novel-часть практически не предсказывает активность сама по себе.')

if res_full['rmse'] < min(res_dist['rmse'], res_nov['rmse']) - 0.02:
    print('  -> distilled и novel ДОПОЛНЯЮТ друг друга (синергия).')
elif abs(res_full['rmse'] - res_dist['rmse']) < 0.05:
    print('  -> distilled-части практически достаточно; novel почти не вносит вклада.')
else:
    print('  -> distilled-часть несёт основной сигнал, novel — небольшой добавок.')


lines = [
    'ВЕРДИКТ',
    '=======',
    '[A] {} координат полностью объяснимы из MolFormer (R2>=0.9),'.format(int(is_distilled.sum())),
    '    {} частично (0.5-0.9), {} новые (R2<0.5).'.format(int(is_partial.sum()), int(is_novel.sum())),
    '    Доля дисперсии emb: distilled={:.3f}, novel={:.3f}.'.format(share_distilled_in, share_novel_in),
    '',
    '[B] SHAP по типам (sum / mean / count):',
    shap_by_type.to_string(),
    '',
    '[C] OOD downstream:',
    '  emb_full       RMSE={:.3f}  R2={:.3f}'.format(res_full['rmse'], res_full['r2']),
    '  emb_distilled  RMSE={:.3f}  R2={:.3f}'.format(res_dist['rmse'], res_dist['r2']),
    '  emb_novel      RMSE={:.3f}  R2={:.3f}'.format(res_nov ['rmse'], res_nov ['r2']),
]
with open(OUT_DIR / 'VERDICT_decomposition.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print('\n✓', OUT_DIR / 'VERDICT_decomposition.txt')
print('\n✨ ГОТОВО!')
